## Silver Layer - Class-Based Transformation

This notebook uses a single **`SilverTransformation`** class to clean and standardise all Bronze tables into Silver.

Every entity follows the same pipeline:
1. **Read** the Bronze Delta table
2. **Drop** unnecessary columns (e.g. `url`)
3. **Rename** columns to snake_case for consistency
4. **Deduplicate** on natural keys
5. **InitCap** string columns for uniform casing
6. **Write** to a Silver Delta table with `overwriteSchema`

Helper methods (`drop`, `rename`, `notnull`, `dropduplicates`, `initcap`, `write`) keep the transform methods short and readable — each entity method only defines what is *different* (which columns to drop, rename, etc.).

In [0]:
catalog = 'formula1'
silver_schema ="practice_class_silver"
bronze_schema ="practice_class" 

In [0]:
from pyspark.sql import functions as F

class SilverTransformation:

    def __init__(self, catalog, bronze_schema, silver_schema, spark):
        self.spark = spark
        self.catalog = catalog
        self.bronze_schema = bronze_schema
        self.silver_schema = silver_schema

    def bronze_table(self, name):
        return f"{self.catalog}.{self.bronze_schema}.{name}"

    def silver_table(self, name):
        return f"{self.catalog}.{self.silver_schema}.{name}"

    def read(self, table_name):
        return self.spark.read.table(table_name)

    def drop(self, df, columns):
        return df.drop(*columns)

    def rename(self, df, rename_map):
        return df.withColumnsRenamed(rename_map)

    def notnull(self, df, columns):
        return df.dropna(subset=columns)

    def dropduplicates(self, df, columns):
        return df.dropDuplicates(columns)

    def initcap(self, df, columns):
        for c in columns:
            df = df.withColumn(c, F.initcap(F.col(c)))
        return df

    def write(self, df, table_name):
        (
            df.write
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .format('delta')
            .saveAsTable(table_name)
        )

    def transform_circuits(self):
        df = self.read(self.bronze_table('circuits_practice'))
        df = self.drop(df, ['url'])
        df = self.rename(df, {
            'circuitId':   'circuit_id',
            'circuitName': 'circuit_name',
            'lat':         'latitude',
            'long':        'longitude'
        })
        df = self.notnull(df, ['circuit_id'])
        df = self.dropduplicates(df, ['circuit_id'])
        df = self.initcap(df, ['circuit_name', 'locality'])
        self.write(df, self.silver_table('circuits_practice_class'))
    def transform_races(self):
        df = self.read(self.bronze_table('races_practice'))
        df = self.drop(df,['url'])
        df = self.rename(df,{
            'raceId': 'race_id',
            'raceName': 'race_name',
            'date': 'race_date',
            'circuitId': 'circuit_id'
        })
        df = self.dropduplicates(df,['season','round'])
        df = self.initcap(df,['race_name'])
        self.write(df, self.silver_table('races_practice_class'))
    def transform_constructors(self):
        df = self.read(self.bronze_table('constructors_practice'))
        df = self.drop(df,['url'])
        df = self.rename(df,{
            'constructorId': 'constructor_id',
            'constructorRef': 'constructor_ref',
            'name' : 'constructor_name'
        })
        df = self.dropduplicates(df,['constructor_id'])
        df = self.initcap(df,['constructor_name'])
        self.write(df,self.silver_table('constructors_practice_class'))
    def transform_results(self):
        df = self.read(self.bronze_table('results_practice'))
        df = self.drop(df,['fastestLapSpeed'])
        df = self.rename(df, {
            'constructorId': 'constructor_id',
            'driverId':      'driver_id',
            'raceName':      'race_name',
            'date':          'race_date',
            'grid':          'grid_position',
            'laps':          'completed_laps',
            'number':        'car_number',
            'position':      'final_position',
            'positionText':  'final_position_text'
        })
        df = self.notnull(df,['constructor_id','driver_id','grid_position','completed_laps','car_number','final_position','final_position_text'])
        df = self.dropduplicates(df,['season','round','driver_id'])
        df = self.initcap(df,['race_name','final_position_text'])
        self.write(df,self.silver_table('results_practice_class'))
    def transform_drivers(self):
        df = self.read(self.bronze_table('drivers_practice'))
        df = self.drop(df,['URL'])
        df = df.withColumn('driver_name', F.concat_ws(' ', F.col('name.givenName'), F.col('name.familyName')))
        df = df.drop('name')
        df = self.rename(df,{
            'driverId': 'driver_id',
            'driverRef': 'driver_ref',
            'dateOfBirth': 'date_of_birth',
            'nationality': 'nationality'
        })
        df = self.dropduplicates(df,['driver_id'])
        df = self.initcap(df,['driver_name','nationality'])
        self.write(df,self.silver_table('drivers_practice_class'))
    def transform_sprints(self):
        df = self.read(self.bronze_table('sprints_practice'))
        df = self.drop(df,['url'])
        df = self.rename(df,{
            'constructorId': 'constructor_id',
            'driverId':      'driver_id',
            'raceName':      'race_name',
            'date':          'race_date',
            'grid':          'grid_position',
            'laps':          'completed_laps',
            'number':        'car_number',
            'position':      'final_position',
            'positionText':  'final_position_text'
        })
        df = self.dropduplicates(df,['season','round','driver_id'])
        df = self.initcap(df,['race_name','final_position_text'])
        self.write(df,self.silver_table('sprints_practice_class'))

    def transform_all(self):
        self.transform_circuits()
        self.transform_races()
        self.transform_constructors()
        self.transform_drivers()
        self.transform_results()
        self.transform_sprints()
        print("All Silver tables transformed successfully.")

silver = SilverTransformation(
    spark=spark,
    catalog=catalog,
    bronze_schema=bronze_schema,
    silver_schema=silver_schema
)

silver.transform_all()